# Индексация MT-RAG через Elasticsearch + ELSER

```text
JSONL -> документы Elasticsearch -> ELSER sparse embeddings -> snapshot -> ZIP
```

ZIP потом восстанавливается в локальный Elasticsearch

## 1. Python-клиент Elasticsearch

In [1]:
!pip install -q "elasticsearch>=9,<10"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.6/993.6 kB 19.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 3.7 MB/s eta 0:00:00


## 2. Чтение одного корпуса

In [2]:
import json
from pathlib import Path
import pandas as pd

data_dir = Path("/kaggle/input/datasets/zvictor12/mt-rag")

def load_passages(domain):
    """Загрузить один JSONL-корпус в таблицу для индексации."""
    records = []

    with open(data_dir / f"{domain}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            text = (item.get("text") or "").strip()
            if not text:
                continue

            records.append({
                "row_idx": len(records),
                "id": item.get("_id") or item["id"],
                "title": item.get("title", ""),
                "text": text,
                "url": item.get("url", ""),
            })

    return pd.DataFrame.from_records(records, index="row_idx")

## 3. Скачивание и распаковка Elasticsearch

In [3]:
from urllib.request import urlretrieve
import shutil

es_version = "9.4.3"

es_root = Path("/kaggle/temp/elasticsearch") # папка elasticsearch
es_home = es_root / f"elasticsearch-{es_version}" # подпапка в установленном клиенте
es_archive = es_root / f"elasticsearch-{es_version}.tar.gz" # путь для установки архива
snapshot_dir = Path("/kaggle/working/elser-snapshot") # сохранение индексов

es_root.mkdir(parents=True, exist_ok=True)
snapshot_dir.mkdir(parents=True, exist_ok=True)

if not es_home.exists():
    url = f"https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-{es_version}-linux-x86_64.tar.gz"
    urlretrieve(url, es_archive)
    shutil.unpack_archive(es_archive, es_root)

    es_archive.unlink()

## 4. Запуск временного Elasticsearch

In [4]:
import os
import subprocess
import requests
import time

es_url = "http://127.0.0.1:9200"

command = [str(es_home / "bin/elasticsearch")]

# Elasticsearch нельзя запускать от root
if os.geteuid() == 0:
    subprocess.run(["useradd", "-m", "esuser"], stderr=subprocess.DEVNULL)
    subprocess.run(
        ["chown", "-R", "esuser:esuser", str(es_root), str(snapshot_dir)],
        check=True,
    )
    command = ["runuser", "-u", "esuser", "--", *command]

command.extend([
    "-Ediscovery.type=single-node",
    "-Expack.security.enabled=false",
    "-Expack.ml.max_machine_memory_percent=60",
    "-Ehttp.host=127.0.0.1",
    f"-Epath.repo={snapshot_dir}",
])

environment = os.environ.copy()
environment["ES_JAVA_OPTS"] = "-Xms2g -Xmx2g"

log = open(es_root / "elasticsearch.log", "w")
es_process = subprocess.Popen(command, env=environment, stdout=log, stderr=subprocess.STDOUT)

while True:
    if es_process.poll() is not None:
        raise RuntimeError((es_root / "elasticsearch.log").read_text())
    try:
        if requests.get(es_url, timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)

print(requests.get(es_url).json()["version"]["number"])

9.4.3


## 5. Доступ к API и trial-лицензия

In [5]:
from elasticsearch import Elasticsearch, helpers

def es_request(method, path, body=None, **kwargs):
    """Выполнить JSON REST-запрос к Elasticsearch."""
    response = requests.request(method, es_url + path, json=body, **kwargs)

    response.raise_for_status()
    if response.content:
        return response.json()



es = Elasticsearch(
    es_url,
    request_timeout=900,
    retry_on_timeout=True,
    retry_on_status=(429, 502, 503, 504),
    max_retries=8,
)

print(es_request(
    "POST",
    "/_license/start_trial",
    params={"acknowledge": "true"},
))

{'acknowledged': True, 'trial_was_started': True, 'type': 'trial'}


## 6. Развёртывание ELSER

In [6]:
inference_id = "mtrag-elser"
model_id = ".elser_model_2_linux-x86_64"
allocations = 4

es_request(
    "PUT",
    f"/_inference/sparse_embedding/{inference_id}",
    {
        "service": "elasticsearch",
        "service_settings": {
            "model_id": model_id,
            # Число CPU-потоков внутри одной allocation.
            "num_threads": 1,
            "adaptive_allocations": {
                "enabled": True,
                "min_number_of_allocations": allocations,
                "max_number_of_allocations": allocations,
            },
        },
        "chunking_settings": {"strategy": "none"},
    },
    params={"timeout": "30m"},
    timeout=31 * 60,
)


while True:
    response = requests.post(
        f"{es_url}/_inference/sparse_embedding/{inference_id}",
        json={"input": "test query"},
        timeout=120,
    )
    if response.ok:
        break
    time.sleep(15)

print("ELSER ready")

stats = es_request("GET", f"/_ml/trained_models/{model_id}/_stats")
deployment = stats["trained_model_stats"][0].get("deployment_stats", {})
print("state:", deployment.get("state"))
print("allocations:", deployment.get("allocation_status"))
print("threads per allocation:", deployment.get("threads_per_allocation"))

ELSER ready
state: started
allocations: {'allocation_count': 4, 'target_allocation_count': 4, 'state': 'fully_allocated'}
threads per allocation: 1


## 7. Bulk-индексация

In [7]:
def index_name(domain):
    """Сформировать имя индекса для корпуса"""
    return f"mtrag-{domain}-elser"


def create_index(domain):
    """Создать индекс корпуса, не удаляя уже выполненную часть"""
    name = index_name(domain)

    if es.indices.exists(index=name):
        return name

    es.indices.create(
        index=name,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "-1",
        },
        mappings={
            "properties": {
                "row_idx": {"type": "integer"},
                "doc_id": {"type": "keyword"},
                "title": {"type": "text"},
                "text": {"type": "text", "copy_to": "semantic_text"},
                "url": {"type": "keyword", "ignore_above": 2048},
                "semantic_text": {
                    "type": "semantic_text",
                    "inference_id": inference_id,
                    "chunking_settings": {"strategy": "none"},
                },
            }
        },
    )

    return name

In [8]:
from tqdm.auto import tqdm

bulk_size = 64
bulk_workers = 2

def index_domain(domain):
    """Проиндексировать все passages одного корпуса через ELSER."""
    passages = load_passages(domain)
    name = create_index(domain)

    es.indices.refresh(index=name)
    existing_ids = {
        hit["_id"]
        for hit in helpers.scan(
            es,
            index=name,
            query={"query": {"match_all": {}}},
            source=False,
            size=2000,
            request_timeout=900,
        )
    }

    print(domain, "already indexed:", len(existing_ids))

    actions = (
        {
            "_index": name,
            "_id": str(row.id),
            "_source": {
                "row_idx": row.Index,
                "doc_id": row.id,
                "title": row.title,
                "text": row.text,
                "url": row.url,
            },
        }
        for row in passages.itertuples()
        if str(row.id) not in existing_ids
    )

    stream = helpers.parallel_bulk(
        es,
        actions,
        thread_count=bulk_workers,
        queue_size=bulk_workers,
        chunk_size=bulk_size,
    )

    with tqdm(total=len(passages), initial=len(existing_ids), desc=domain) as progress:
        for ok, item in stream:
            if not ok:
                raise RuntimeError(item)
            progress.update()

    es.indices.refresh(index=name)
    print(name, es.count(index=name)["count"])


repository = "mtrag-repository"

es_request(
    "PUT",
    f"/_snapshot/{repository}",
    {
        "type": "fs",
        "settings": {"location": str(snapshot_dir), "compress": True},
    },
)


def save_domain_checkpoint(domain):
    """Сохранить готовый домен в snapshot и обновить checkpoint ZIP."""
    checkpoint = f"mtrag-elser-{domain}"
    path = f"/_snapshot/{repository}/{checkpoint}"

    if not requests.get(es_url + path, timeout=30).ok:
        es_request(
            "PUT",
            path,
            {
                "indices": index_name(domain),
                "include_global_state": False,
            },
            params={"wait_for_completion": "true"},
            timeout=3600,
        )

    checkpoint_zip = shutil.make_archive(
        "/kaggle/working/mtrag_elser_checkpoint",
        "zip",
        root_dir=snapshot_dir,
    )
    print("checkpoint:", checkpoint_zip)

domains = ["fiqa"]

def run_all_computations():
    """Последовательно проиндексировать четыре корпуса."""
    for domain in domains:
        index_domain(domain)
        save_domain_checkpoint(domain)

## 10. Запуск индексации

In [ ]:
run_all_computations()

cloud already indexed: 0


cloud:   0%|          | 0/72439 [00:00<?, ?it/s]

## 11. Snapshot и ZIP для скачивания

In [ ]:
repository = "mtrag-repository"
snapshot = "mtrag-elser"

es_request(
    "PUT",
    f"/_snapshot/{repository}",
    {
        "type": "fs",
        "settings": {
            "location": str(snapshot_dir),
            "compress": True,
        },
    },
)

es_request(
    "PUT",
    f"/_snapshot/{repository}/{snapshot}",
    {
        "indices": ",".join(index_name(domain) for domain in domains),
        "include_global_state": False,
    },
    params={"wait_for_completion": "true"},
    timeout=3600,
)

es_request("DELETE", f"/_snapshot/{repository}")

archive_path = shutil.make_archive(
    "/kaggle/working/mtrag_elser_snapshot",
    "zip",
    root_dir=snapshot_dir,
)

Path("/kaggle/working/mtrag_elser_checkpoint.zip").unlink(missing_ok=True)
print(archive_path)